# Phase 1F reference notebook

This notebook retains the recorded experiment workflow. Start a fresh Jupyter kernel in a Linux GPU environment with the prerequisites in [Reproduction guide](../docs/reproducibility.md). Run the dataset preparation CLI first. The recorded workflow derives its nnU-Net plan from all converted cases. Its results are historical validation measurements. Keep training outputs and case-level figures local.


Start the notebook from the project root or the notebooks directory. The next setup cell also accepts the GLIOMA_PROJECT_ROOT environment variable.


In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import sys
from datetime import datetime

def run_live(cmd, cwd=None, log_path=None):
    print(">>", " ".join(map(str, cmd)))
    log_file = None
    try:
        if log_path is not None:
            log_path = Path(log_path)
            log_path.parent.mkdir(parents=True, exist_ok=True)
            log_file = log_path.open("a", encoding="utf-8")
            log_file.write("\n\n=== COMMAND ===\n")
            log_file.write(" ".join(map(str, cmd)) + "\n")
            log_file.flush()

        p = subprocess.Popen(
            cmd,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in p.stdout:
            print(line, end="")
            if log_file is not None:
                log_file.write(line)
                log_file.flush()

        p.wait()
        if p.returncode != 0:
            raise RuntimeError(f"Komut hata verdi: {p.returncode}")
        return p
    finally:
        if log_file is not None:
            log_file.close()

def ensure_nnunet_cli(cmd_name: str):
    found = shutil.which(cmd_name)
    if not found:
        raise RuntimeError(f"{cmd_name} is missing. Install nnU-Net v2 in the Jupyter environment first.")
    print(f"[OK] {cmd_name}: {found}")


def copy_if_exists(src: Path, dst: Path):
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("copied:", src, "->", dst)
    else:
        print("[WARN] missing:", src)


In [ ]:
# fresh: yeni 1000 epoch run
# resume: ayni RUN_NAME altindaki checkpointten devam
START_MODE = "fresh"  # Set to "resume" to reuse this run name.

# fresh icin yeni isim ver.
# resume icin ayni ismi koru.
RUN_NAME = "phase1f_fold0_full_run_001"

project_hint = os.environ.get("GLIOMA_PROJECT_ROOT")
PROJECT_ROOT = Path(project_hint).expanduser().resolve() if project_hint else Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "env_check.py").is_file():
    raise FileNotFoundError("Start in the project root or set GLIOMA_PROJECT_ROOT.")
SOURCE_DATASET_ROOT = PROJECT_ROOT / "outputs" / "phase1_nnunet_dataset"

RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
MONITOR_DIR = RUN_ROOT / "monitoring"

WORK_ROOT = PROJECT_ROOT / "outputs" / "workspace" / RUN_NAME
CONTENT_DATASET_ROOT = WORK_ROOT / "outputs" / "phase1_nnunet_dataset"

DATASET_ID = 901
DATASET_NAME = "Phase1GliomaBaseline"
CONFIG_NAME = "3d_fullres"
DATASET_FOLDER = f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"

print("START_MODE:", START_MODE)
print("RUN_NAME:", RUN_NAME)
print("SOURCE_DATASET_ROOT:", SOURCE_DATASET_ROOT)
print("RUN_ROOT:", RUN_ROOT)


In [ ]:
NNUNET_RAW = WORK_ROOT / "nnunet_raw"
NNUNET_PREPROCESSED = WORK_ROOT / "nnunet_preprocessed"
NNUNET_RESULTS = RUN_ROOT / "nnunet_results"

for d in [WORK_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, CONTENT_DATASET_ROOT, MONITOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])


In [ ]:
required = [
    SOURCE_DATASET_ROOT / "case_id_map.csv",
    SOURCE_DATASET_ROOT / "split_summary.json",
    SOURCE_DATASET_ROOT / "train_case_ids.txt",
    SOURCE_DATASET_ROOT / "val_case_ids.txt",
    SOURCE_DATASET_ROOT / "held_out_test_case_ids.txt",
    SOURCE_DATASET_ROOT / "splits_final.json",
    SOURCE_DATASET_ROOT / "dataset.json",
    SOURCE_DATASET_ROOT / DATASET_FOLDER / "dataset.json",
    SOURCE_DATASET_ROOT / DATASET_FOLDER / "imagesTr",
    SOURCE_DATASET_ROOT / DATASET_FOLDER / "labelsTr",
]

for p in required:
    print(p, "exists=", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"Eksik: {p}")

split_summary = json.loads((SOURCE_DATASET_ROOT / "split_summary.json").read_text(encoding="utf-8"))
print("leakage_check:", split_summary.get("leakage_check"))
if not split_summary.get("leakage_check", {}).get("pass", False):
    raise RuntimeError("Leakage PASS degil. Traininge gecme.")

run_live(
    ["rsync", "-a", "--delete", str(SOURCE_DATASET_ROOT) + "/", str(CONTENT_DATASET_ROOT) + "/"],
    log_path=MONITOR_DIR / "stage_rsync.log",
)

raw_dataset_dir = NNUNET_RAW / DATASET_FOLDER
if raw_dataset_dir.exists() or raw_dataset_dir.is_symlink():
    if raw_dataset_dir.is_dir() and not raw_dataset_dir.is_symlink():
        shutil.rmtree(raw_dataset_dir)
    else:
        raw_dataset_dir.unlink()

raw_dataset_dir.symlink_to(CONTENT_DATASET_ROOT / DATASET_FOLDER, target_is_directory=True)

img_count = len(list((CONTENT_DATASET_ROOT / DATASET_FOLDER / "imagesTr").glob("*.nii.gz")))
lbl_count = len(list((CONTENT_DATASET_ROOT / DATASET_FOLDER / "labelsTr").glob("*.nii.gz")))
print("imagesTr:", img_count, "labelsTr:", lbl_count)


In [ ]:
drive_root_json = SOURCE_DATASET_ROOT / "dataset.json"
drive_dataset_json = SOURCE_DATASET_ROOT / DATASET_FOLDER / "dataset.json"
content_root_json = CONTENT_DATASET_ROOT / "dataset.json"
content_dataset_json = CONTENT_DATASET_ROOT / DATASET_FOLDER / "dataset.json"
raw_dataset_json = NNUNET_RAW / DATASET_FOLDER / "dataset.json"

for p in [drive_root_json, drive_dataset_json, content_root_json, content_dataset_json, raw_dataset_json]:
    print("\n==", p)
    text = p.read_text(encoding="utf-8")
    print(text)
    if '"label_4": 4' not in text:
        raise RuntimeError(f"Beklenen multiclass label contract yok: {p}")


In [ ]:
static_files = [
    "split_manifest_phase1.csv",
    "split_summary.json",
    "train_case_ids.txt",
    "val_case_ids.txt",
    "held_out_test_case_ids.txt",
    "splits_final.json",
    "training_plan.json",
    "training_commands.txt",
    "training_checklist.md",
    "dataset.json",
]

for name in static_files:
    copy_if_exists(SOURCE_DATASET_ROOT / name, RUN_ROOT / name)

copy_if_exists(SOURCE_DATASET_ROOT / DATASET_FOLDER / "dataset.json", RUN_ROOT / DATASET_FOLDER / "dataset.json")


In [ ]:
ensure_nnunet_cli("nnUNetv2_plan_and_preprocess")
ensure_nnunet_cli("nnUNetv2_train")
print("[DONE] nnU-Net CLI hazir.")


In [ ]:
if START_MODE not in {"fresh", "resume"}:
    raise RuntimeError("START_MODE yalnizca 'fresh' veya 'resume' olabilir.")

prep_dataset_dir = NNUNET_PREPROCESSED / DATASET_FOLDER
results_dataset_dir = NNUNET_RESULTS / DATASET_FOLDER

if START_MODE == "fresh" and results_dataset_dir.exists():
    raise FileExistsError("This run already has model outputs. Choose a new RUN_NAME or use resume mode.")

# Preprocessed her yeni runtime'da temiz uretilebilir.
if prep_dataset_dir.exists():
    shutil.rmtree(prep_dataset_dir)
    print("removed preprocessed:", prep_dataset_dir)
else:
    print("preprocessed not found:", prep_dataset_dir)


In [ ]:
raw_dataset_dir = NNUNET_RAW / DATASET_FOLDER
prep_dataset_dir = NNUNET_PREPROCESSED / DATASET_FOLDER
plans_json = prep_dataset_dir / "nnUNetPlans.json"

print("raw_dataset_dir:", raw_dataset_dir, "exists:", raw_dataset_dir.exists())
if not raw_dataset_dir.exists():
    raise FileNotFoundError(f"Raw dataset yok: {raw_dataset_dir}")

run_live(
    ["nnUNetv2_plan_and_preprocess", "-d", str(DATASET_ID), "-c", CONFIG_NAME],
    log_path=MONITOR_DIR / "planner_preprocess.log",
)

prep_dataset_dir.mkdir(parents=True, exist_ok=True)
src_splits = CONTENT_DATASET_ROOT / "splits_final.json"
dst_splits = prep_dataset_dir / "splits_final.json"
shutil.copy2(src_splits, dst_splits)
print("splits_final.json copied to:", dst_splits)

if not plans_json.exists():
    raise RuntimeError("nnUNetPlans.json olusmadi.")

plans = json.loads(plans_json.read_text(encoding="utf-8"))
cfg = plans.get("configurations", {}).get(CONFIG_NAME, {})
planner_summary = {
    "status": "resolved",
    "configuration_name": CONFIG_NAME,
    "planned_patch_size": cfg.get("patch_size"),
    "planned_batch_size": cfg.get("batch_size"),
    "target_spacing": cfg.get("spacing") or cfg.get("target_spacing") or plans.get("target_spacing"),
    "plans_json": str(plans_json),
}
summary_path = MONITOR_DIR / "planner_inspection_summary.json"
summary_path.write_text(json.dumps(planner_summary, indent=2), encoding="utf-8")
print(json.dumps(planner_summary, indent=2))
print("Wrote:", summary_path)


## Train fold 0


In [ ]:
fold0_dir = NNUNET_RESULTS / DATASET_FOLDER / "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0"
fold0_dir.parent.mkdir(parents=True, exist_ok=True)

if START_MODE == "fresh":
    cmd = ["nnUNetv2_train", str(DATASET_ID), CONFIG_NAME, "0"]
    log_path = MONITOR_DIR / "fold0_full_train.log"
else:
    checkpoint_candidates = [
        fold0_dir / "checkpoint_latest.pth",
        fold0_dir / "checkpoint_final.pth",
        fold0_dir / "checkpoint_best.pth",
    ]
    if not any(p.exists() for p in checkpoint_candidates):
        raise RuntimeError(f"Resume icin checkpoint yok: {fold0_dir}")
    cmd = ["nnUNetv2_train", str(DATASET_ID), CONFIG_NAME, "0", "--c"]
    log_path = MONITOR_DIR / "fold0_resume.log"

run_live(cmd, log_path=log_path)


## Inspect training outputs


In [ ]:
fold0_dir = NNUNET_RESULTS / DATASET_FOLDER / "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0"
print("fold0_dir exists:", fold0_dir.exists(), fold0_dir)

if fold0_dir.exists():
    files = sorted([str(p.relative_to(fold0_dir)) for p in fold0_dir.rglob("*") if p.is_file()])
    print("file count:", len(files))
    print("sample files:", files[:80])

    important = [
        fold0_dir / "checkpoint_latest.pth",
        fold0_dir / "checkpoint_best.pth",
        fold0_dir / "checkpoint_final.pth",
    ]
    for p in important:
        print(p.name, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else -1)


## Phase 1F Validation Evaluation
Bu blok held-out test kullanmaz.
Yalnizca validation split uzerinde:
- prediction
- sinif bazli dice
- hacim karsilastirma
- presence/classification ozeti
- renkli 2D tri-planar goruntuler
uretir.


### Import evaluation dependencies


In [ ]:
import json
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 5)


### Stage the validation cases


In [ ]:
RUN_EVAL_DIR = RUN_ROOT / "validation_eval"
VAL_INPUT_DIR = WORK_ROOT / "validation_eval_input"
VAL_GT_DIR = WORK_ROOT / "validation_eval_gt"
PRED_DIR = RUN_EVAL_DIR / "predictions_fold0"
FIG_DIR = RUN_EVAL_DIR / "figures"
SUMMARY_DIR = RUN_EVAL_DIR / "summary"

for d in [RUN_EVAL_DIR, VAL_INPUT_DIR, VAL_GT_DIR, PRED_DIR, FIG_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

val_case_ids = [
    line.strip()
    for line in (SOURCE_DATASET_ROOT / "val_case_ids.txt").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print("validation case count:", len(val_case_ids))
print("sample:", val_case_ids[:10])

fold0_dir = NNUNET_RESULTS / DATASET_FOLDER / "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0"
print("fold0_dir:", fold0_dir, "exists=", fold0_dir.exists())

checkpoint_name = None
for name in ["checkpoint_best.pth", "checkpoint_final.pth", "checkpoint_latest.pth"]:
    p = fold0_dir / name
    if p.exists():
        checkpoint_name = name
        print("selected checkpoint:", checkpoint_name)
        break

if checkpoint_name is None:
    raise FileNotFoundError(f"Checkpoint yok: {fold0_dir}")

images_tr = SOURCE_DATASET_ROOT / DATASET_FOLDER / "imagesTr"
labels_tr = SOURCE_DATASET_ROOT / DATASET_FOLDER / "labelsTr"

# input/gt klasorlerini temizle ve yeniden kur
for d in [VAL_INPUT_DIR, VAL_GT_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

for case_id in val_case_ids:
    for ch in range(4):
        src = images_tr / f"{case_id}_{ch:04d}.nii.gz"
        dst = VAL_INPUT_DIR / src.name
        if not src.exists():
            raise FileNotFoundError(src)
        dst.symlink_to(src)

    gt_src = labels_tr / f"{case_id}.nii.gz"
    gt_dst = VAL_GT_DIR / gt_src.name
    if not gt_src.exists():
        raise FileNotFoundError(gt_src)
    gt_dst.symlink_to(gt_src)

print("VAL_INPUT_DIR files:", len(list(VAL_INPUT_DIR.glob("*.nii.gz"))))
print("VAL_GT_DIR files:", len(list(VAL_GT_DIR.glob("*.nii.gz"))))


### Predict full-volume masks


In [ ]:
predict_cmd = [
    "nnUNetv2_predict",
    "-d", str(DATASET_ID),
    "-i", str(VAL_INPUT_DIR),
    "-o", str(PRED_DIR),
    "-f", "0",
    "-c", CONFIG_NAME,
    "-chk", checkpoint_name,
]

run_live(
    predict_cmd,
    log_path=RUN_EVAL_DIR / "predict_validation.log",
)


### Score each validation case


In [ ]:
dataset_json = json.loads((SOURCE_DATASET_ROOT / "dataset.json").read_text(encoding="utf-8"))
label_map = dataset_json["labels"]
foreground_labels = {k: v for k, v in label_map.items() if k != "background"}
label_ids = sorted(foreground_labels.values())
id_to_name = {v: k for k, v in foreground_labels.items()}

def dice_score(gt_mask, pr_mask):
    gt_sum = int(gt_mask.sum())
    pr_sum = int(pr_mask.sum())
    if gt_sum == 0 and pr_sum == 0:
        return np.nan
    inter = int((gt_mask & pr_mask).sum())
    return (2.0 * inter) / (gt_sum + pr_sum)

rows = []

for case_id in val_case_ids:
    gt_path = VAL_GT_DIR / f"{case_id}.nii.gz"
    pr_path = PRED_DIR / f"{case_id}.nii.gz"
    img_path = images_tr / f"{case_id}_0000.nii.gz"

    if not pr_path.exists():
        print("[WARN] missing prediction:", pr_path)
        continue

    gt_img = nib.load(str(gt_path))
    pr_img = nib.load(str(pr_path))

    gt = np.asarray(gt_img.get_fdata(), dtype=np.int16)
    pr = np.asarray(pr_img.get_fdata(), dtype=np.int16)

    if gt.shape != pr.shape:
        raise RuntimeError(f"Shape mismatch for {case_id}: {gt.shape} vs {pr.shape}")

    voxel_volume_ml = float(np.prod(gt_img.header.get_zooms()[:3])) / 1000.0

    gt_fg = gt > 0
    pr_fg = pr > 0

    case_row = {
        "case_id": case_id,
        "foreground_dice": dice_score(gt_fg, pr_fg),
        "gt_foreground_volume_ml": float(gt_fg.sum()) * voxel_volume_ml,
        "pred_foreground_volume_ml": float(pr_fg.sum()) * voxel_volume_ml,
    }

    for label_id in label_ids:
        name = id_to_name[label_id]
        gt_mask = gt == label_id
        pr_mask = pr == label_id

        gt_present = bool(gt_mask.any())
        pr_present = bool(pr_mask.any())

        gt_vol = float(gt_mask.sum()) * voxel_volume_ml
        pr_vol = float(pr_mask.sum()) * voxel_volume_ml
        vol_err = pr_vol - gt_vol

        case_row[f"{name}_dice"] = dice_score(gt_mask, pr_mask)
        case_row[f"{name}_gt_present"] = gt_present
        case_row[f"{name}_pred_present"] = pr_present
        case_row[f"{name}_gt_volume_ml"] = gt_vol
        case_row[f"{name}_pred_volume_ml"] = pr_vol
        case_row[f"{name}_volume_error_ml"] = vol_err
        case_row[f"{name}_volume_abs_error_ml"] = abs(vol_err)

    rows.append(case_row)

metrics_df = pd.DataFrame(rows).sort_values("foreground_dice", ascending=False).reset_index(drop=True)
metrics_csv = RUN_EVAL_DIR / "validation_metrics_per_case.csv"
metrics_df.to_csv(metrics_csv, index=False)

display(metrics_df.head(10))
print("saved:", metrics_csv)
print("rows:", len(metrics_df))


### Summarize label metrics


In [ ]:
summary_rows = []
presence_rows = []

for label_id in label_ids:
    name = id_to_name[label_id]
    dice_col = f"{name}_dice"
    gt_present_col = f"{name}_gt_present"
    pred_present_col = f"{name}_pred_present"
    gt_vol_col = f"{name}_gt_volume_ml"
    pred_vol_col = f"{name}_pred_volume_ml"
    abs_err_col = f"{name}_volume_abs_error_ml"

    active_mask = metrics_df[gt_present_col] | metrics_df[pred_present_col]
    active = metrics_df.loc[active_mask]

    tp = int(((metrics_df[gt_present_col]) & (metrics_df[pred_present_col])).sum())
    fp = int((~metrics_df[gt_present_col] & metrics_df[pred_present_col]).sum())
    fn = int((metrics_df[gt_present_col] & ~metrics_df[pred_present_col]).sum())
    tn = int((~metrics_df[gt_present_col] & ~metrics_df[pred_present_col]).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = 2 * precision * recall / (precision + recall) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan

    summary_rows.append({
        "label_name": name,
        "label_id": label_id,
        "n_cases": int(len(metrics_df)),
        "n_active_cases": int(len(active)),
        "mean_dice_active": float(active[dice_col].dropna().mean()) if len(active) else np.nan,
        "median_dice_active": float(active[dice_col].dropna().median()) if len(active) else np.nan,
        "mean_gt_volume_ml": float(metrics_df[gt_vol_col].mean()),
        "mean_pred_volume_ml": float(metrics_df[pred_vol_col].mean()),
        "mean_abs_volume_error_ml": float(metrics_df[abs_err_col].mean()),
        "presence_precision": precision,
        "presence_recall": recall,
        "presence_f1": f1,
    })

    presence_rows.append({
        "label_name": name,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

summary_df = pd.DataFrame(summary_rows)
presence_df = pd.DataFrame(presence_rows)

summary_df.to_csv(SUMMARY_DIR / "label_summary.csv", index=False)
presence_df.to_csv(SUMMARY_DIR / "presence_classification_summary.csv", index=False)

validation_summary = {
    "checkpoint_used": checkpoint_name,
    "n_validation_cases": int(len(metrics_df)),
    "mean_foreground_dice": float(metrics_df["foreground_dice"].mean()),
    "median_foreground_dice": float(metrics_df["foreground_dice"].median()),
    "best_case_id": str(metrics_df.iloc[0]["case_id"]),
    "worst_case_id": str(metrics_df.iloc[-1]["case_id"]),
}
(SUMMARY_DIR / "validation_summary.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")

display(summary_df)
display(presence_df)
print("saved summary dir:", SUMMARY_DIR)


### Plot validation summaries


In [ ]:
# 1) foreground dice histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(metrics_df["foreground_dice"].dropna(), bins=20, color="#1f77b4", edgecolor="black")
ax.set_title("Validation Foreground Dice Histogram")
ax.set_xlabel("Foreground Dice")
ax.set_ylabel("Case Count")
fig.tight_layout()
fig.savefig(FIG_DIR / "foreground_dice_histogram.png", bbox_inches="tight")
plt.show()

# 2) per-class dice boxplot
dice_cols = [f"{id_to_name[i]}_dice" for i in label_ids]
box_data = [metrics_df[c].dropna().values for c in dice_cols]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(box_data, labels=dice_cols, showmeans=True)
ax.set_title("Per-Class Dice Distribution (Validation)")
ax.set_ylabel("Dice")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "per_class_dice_boxplot.png", bbox_inches="tight")
plt.show()

# 3) presence classification bars
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(presence_df))
w = 0.2
ax.bar(x - 1.5*w, presence_df["tp"], width=w, label="TP")
ax.bar(x - 0.5*w, presence_df["fp"], width=w, label="FP")
ax.bar(x + 0.5*w, presence_df["fn"], width=w, label="FN")
ax.bar(x + 1.5*w, presence_df["tn"], width=w, label="TN")
ax.set_xticks(x)
ax.set_xticklabels(presence_df["label_name"])
ax.set_title("Presence / Classification Summary by Label")
ax.set_ylabel("Case Count")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "presence_classification_bars.png", bbox_inches="tight")
plt.show()

# 4) GT vs Pred volume scatter grid
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for ax, label_id in zip(axes, label_ids):
    name = id_to_name[label_id]
    x = metrics_df[f"{name}_gt_volume_ml"].values
    y = metrics_df[f"{name}_pred_volume_ml"].values
    ax.scatter(x, y, alpha=0.7, s=20)
    lim = max(float(np.max(x)) if len(x) else 1.0, float(np.max(y)) if len(y) else 1.0, 1.0)
    ax.plot([0, lim], [0, lim], linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("GT volume (mL)")
    ax.set_ylabel("Pred volume (mL)")

fig.suptitle("GT vs Pred Volume by Label", y=0.98)
fig.tight_layout()
fig.savefig(FIG_DIR / "volume_scatter_grid.png", bbox_inches="tight")
plt.show()


### Select representative cases


In [ ]:
best_cases = metrics_df.sort_values("foreground_dice", ascending=False).head(12)
worst_cases = metrics_df.sort_values("foreground_dice", ascending=True).head(12)

label1_name = id_to_name[label_ids[0]]
label1_col = f"{label1_name}_dice"
label1_focus = metrics_df[metrics_df[label1_col].notna()].sort_values(label1_col, ascending=True).head(12)

best_cases.to_csv(SUMMARY_DIR / "best_cases_top12.csv", index=False)
worst_cases.to_csv(SUMMARY_DIR / "worst_cases_top12.csv", index=False)
label1_focus.to_csv(SUMMARY_DIR / "label1_focus_worst12.csv", index=False)

display(best_cases[["case_id", "foreground_dice", label1_col]])
display(worst_cases[["case_id", "foreground_dice", label1_col]])
display(label1_focus[["case_id", "foreground_dice", label1_col]])


### Generate private case panels


In [ ]:
CASE_PANEL_DIR = FIG_DIR / "case_panels"
for sub in ["best", "worst", "label1_focus"]:
    (CASE_PANEL_DIR / sub).mkdir(parents=True, exist_ok=True)

label_colors = {
    1: (1.0, 0.15, 0.15),
    2: (0.15, 0.85, 0.2),
    3: (0.15, 0.65, 1.0),
    4: (1.0, 0.8, 0.1),
}

def normalize_img(x):
    x = np.asarray(x, dtype=np.float32)
    p1, p99 = np.percentile(x, [1, 99])
    if p99 <= p1:
        return np.zeros_like(x)
    x = np.clip((x - p1) / (p99 - p1), 0, 1)
    return x

def pick_slice_indices(mask_union):
    if mask_union.any():
        x_idx = int(np.argmax(mask_union.sum(axis=(1, 2))))
        y_idx = int(np.argmax(mask_union.sum(axis=(0, 2))))
        z_idx = int(np.argmax(mask_union.sum(axis=(0, 1))))
    else:
        x_idx = mask_union.shape[0] // 2
        y_idx = mask_union.shape[1] // 2
        z_idx = mask_union.shape[2] // 2
    return x_idx, y_idx, z_idx

def draw_overlay(ax, img2d, gt2d, pr2d, title):
    ax.imshow(img2d.T, cmap="gray", origin="lower")
    for lid in label_ids:
        gt_mask = (gt2d == lid).T
        pr_mask = (pr2d == lid).T
        if gt_mask.any():
            masked = np.ma.masked_where(~gt_mask, gt_mask.astype(float))
            ax.imshow(masked, cmap=ListedColormap([label_colors[lid]]), alpha=0.30, origin="lower")
        if pr_mask.any():
            ax.contour(pr_mask.astype(float), levels=[0.5], colors=[label_colors[lid]], linewidths=1.2, linestyles="--")
    ax.set_title(title)
    ax.axis("off")

def make_case_panel(case_id, out_path):
    img = normalize_img(nib.load(str(images_tr / f"{case_id}_0000.nii.gz")).get_fdata())
    gt = np.asarray(nib.load(str(VAL_GT_DIR / f"{case_id}.nii.gz")).get_fdata(), dtype=np.int16)
    pr = np.asarray(nib.load(str(PRED_DIR / f"{case_id}.nii.gz")).get_fdata(), dtype=np.int16)

    union = (gt > 0) | (pr > 0)
    x_idx, y_idx, z_idx = pick_slice_indices(union)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))

    draw_overlay(axes[0, 0], img[:, :, z_idx], gt[:, :, z_idx], pr[:, :, z_idx], f"Axial z={z_idx}")
    draw_overlay(axes[0, 1], img[:, y_idx, :], gt[:, y_idx, :], pr[:, y_idx, :], f"Coronal y={y_idx}")
    draw_overlay(axes[1, 0], img[x_idx, :, :], gt[x_idx, :, :], pr[x_idx, :, :], f"Sagittal x={x_idx}")

    gt_vols = [metrics_df.loc[metrics_df["case_id"] == case_id, f"{id_to_name[lid]}_gt_volume_ml"].iloc[0] for lid in label_ids]
    pr_vols = [metrics_df.loc[metrics_df["case_id"] == case_id, f"{id_to_name[lid]}_pred_volume_ml"].iloc[0] for lid in label_ids]
    names = [id_to_name[lid] for lid in label_ids]

    x = np.arange(len(names))
    w = 0.35
    axes[1, 1].bar(x - w/2, gt_vols, width=w, label="GT")
    axes[1, 1].bar(x + w/2, pr_vols, width=w, label="Pred")
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(names, rotation=20)
    axes[1, 1].set_ylabel("Volume (mL)")
    axes[1, 1].set_title(f"Volume Compare - {case_id}")
    axes[1, 1].legend()

    fg = metrics_df.loc[metrics_df["case_id"] == case_id, "foreground_dice"].iloc[0]
    l1 = metrics_df.loc[metrics_df["case_id"] == case_id, label1_col].iloc[0]
    fig.suptitle(f"{case_id} | foreground_dice={fg:.4f} | {label1_col}={l1:.4f}", y=0.98)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

for bucket_name, bucket_df in [
    ("best", best_cases),
    ("worst", worst_cases),
    ("label1_focus", label1_focus),
]:
    for case_id in bucket_df["case_id"].tolist():
        out_path = CASE_PANEL_DIR / bucket_name / f"{case_id}.png"
        make_case_panel(case_id, out_path)

print("saved case panels under:", CASE_PANEL_DIR)


### Inspect local output files


In [ ]:
print("validation_eval root:", RUN_EVAL_DIR)
print("summary files:")
for p in sorted(SUMMARY_DIR.glob("*")):
    print(" -", p.name)

print("\nfigure files:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p.name)

sample_pngs = sorted((CASE_PANEL_DIR / "worst").glob("*.png"))[:3]
sample_pngs += sorted((CASE_PANEL_DIR / "best").glob("*.png"))[:3]

for p in sample_pngs:
    print("\nPreview:", p.name)
    display(plt.imread(str(p)))


### Optional local archive


In [ ]:
# Buyuk olabilir. Yalniz istersen calistir.
export_root = Path("/content")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = export_root / f"{RUN_NAME}_{stamp}"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(RUN_ROOT)))
print("ZIP:", zip_path)

from google.colab import files
files.download(str(zip_path))
